# Merchant Normalisation & Transaction Categorisation

Implements the first reusable transaction-enrichment layer
(`finance_analytics.enrichment`):

```
Raw transaction -> Merchant normalisation -> Category assignment -> Enriched transaction
```

This layer is built on the same fixture used throughout notebooks 01-04, and
follows PR-011's Critical Rule: **first determine what the dataset actually
requires.** As this notebook shows, the fixture's raw `merchant` and
`category` values are already reliable — there is no evidence of casing,
punctuation or aliasing problems to fix. So the emphasis here is not
"cleaning up messy data" but building a deterministic, explainable,
reusable *pipeline* around already-good data, one that will still behave
safely (never forcing a guess, never silently merging two different
merchants) on messier data in the future.

**No ML, no LLM, no fuzzy matching** — see `docs/execution/04_ANALYTICAL_FEATURES/PR-011_MERCHANT_NORMALISATION_CATEGORISATION.md`.
Every rule below is deterministic and traceable to either an explicit,
curated table or a directly observed pattern in this dataset.

In [1]:
from dataclasses import asdict
from pathlib import Path

import pandas as pd
import plotly.express as px

from finance_analytics.data.quality import build_quality_report
from finance_analytics.enrichment.categories import (
    CONFIDENCE_BY_METHOD,
    DESCRIPTION_KEYWORD_RULES,
    FALLBACK_CATEGORY,
    KNOWN_CATEGORIES,
    MERCHANT_CATEGORY_RULES,
    categorise,
)
from finance_analytics.enrichment.merchants import MERCHANT_ALIASES, normalise_merchant
from finance_analytics.enrichment.models import enrich_transaction, enrich_transactions
from finance_analytics.io.csv import load_transactions_csv
from finance_analytics.validation.transactions import validate_transactions

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

DATA_PATH = Path("../data/raw/finance_analytics_test_transactions.csv")

In [2]:
transactions = load_transactions_csv(DATA_PATH)
print(f"Raw rows: {len(transactions)}")
transactions

Raw rows: 22


,id,date,amount,currency,description,merchant,category,account
0,1,2026-02-02,-12.50,EUR,Morning coffee,Coffee Corner,Food & Dining,Main Account
1,2,2026-02-03,-54.90,EUR,Weekly groceries,Continente,Groceries,Main Account
2,3,2026-02-05,-29.99,EUR,Monthly subscription,Spotify,Subscriptions,Main Account
3,4,2026-02-07,-82.40,EUR,Dinner with friends,O Pescador,Food & Dining,Main Account
4,5,2026-02-10,-45.00,EUR,Electricity bill,EDP,Utilities,Main Account
5,6,2026-02-12,-18.75,EUR,Pharmacy,Farmácia Central,Health,Main Account
6,7,2026-02-15,-120.00,EUR,Train tickets,CP,Transport,Main Account
7,8,2026-02-18,-642.50,EUR,"Flight, Lisbon to Rome",TAP Air,Travel,Main Account
8,9,2026-02-20,-899.00,EUR,New laptop purchase,MediaMarkt,Shopping,Main Account
9,10,2026-02-22,-8.20,EUR,"Lunch, ""daily menu""",Café Central,Food & Dining,Main Account


## 1. Data Quality Assessment

Unlike notebooks 02-04, this notebook does **not** drop the duplicate
transaction ID, the structurally-invalid rows, or the QA fixture rows
before analysis. PR-011's own subject is how the enrichment layer behaves
on exactly this kind of imperfect data ("Do not silently discard
problematic rows") — dropping them first would hide the cases the layer
exists to handle safely. The general data-quality report below (from
PR-007) is unaffected by that choice; it already reports on the full raw
frame.

In [3]:
validation = validate_transactions(transactions)
quality = build_quality_report(transactions)

print("Invalid date rows:", validation.invalid_date_rows)
print("Invalid amount rows:", validation.invalid_amount_rows)
print("Missing-value rows:", validation.missing_value_rows)
print()
quality.to_frame()

Invalid date rows: (19,)
Invalid amount rows: (20,)
Missing-value rows: {'merchant': (21,)}



,value
Rows,22
Columns,8
Duplicate rows,1
Duplicate transaction IDs,1
Invalid dates,1
Invalid amounts,1
Unique merchants,16
Unique categories,10
Date range,2026-02-02 to 2026-03-19
Income transactions,2


**Observed**, specific to merchant/category quality (PR-011, section 5):

- **Missing merchants:** 1 row (`transaction_id=21`) — `merchant` is blank.
  Its `description` ("Missing merchant test") and `category` ("Shopping")
  are both present.
- **Blank descriptions:** none.
- **Casing inconsistencies in `merchant`:** none — every raw merchant value
  is already consistently cased across all of its occurrences (checked
  below, section 2).
- **Punctuation differences in `merchant`:** none observed (no ".com"/"www."
  suffixes, no stray symbols).
- **Obvious merchant aliases:** none observed — every merchant name in this
  fixture is spelled and formatted exactly one way.
- **Unknown categories:** none — every non-blank `category` value already
  belongs to the taxonomy in section 5.
- **Conflicting category assignments:** none — checked directly below,
  section 5.

Two rows (`id=19`, `id=20`) have an invalid date/amount but a perfectly
valid `merchant`/`category`/`description` — evidence that merchant
normalisation and categorisation should not depend on date/amount validity
at all (see `enrichment.models.enrich_transactions`'s docstring).

## 2. Merchant Inconsistencies

Checking directly for the casing/formatting inconsistencies PR-011 asks
about, rather than assuming either way.

In [4]:
raw_merchants = transactions["merchant"].dropna()

exact_unique = raw_merchants.nunique()
casefolded_unique = raw_merchants.str.casefold().nunique()

print(f"Unique raw merchant strings (exact):      {exact_unique}")
print(f"Unique raw merchant strings (casefolded): {casefolded_unique}")
print(
    "-> equal, so no two raw merchant strings differ only by case "
    "(no casing inconsistency exists in this fixture)."
)

Unique raw merchant strings (exact):      16
Unique raw merchant strings (casefolded): 16
-> equal, so no two raw merchant strings differ only by case (no casing inconsistency exists in this fixture).


In [5]:
# A merchant with >1 distinct category is a conflicting assignment; PR-011
# section 5 explicitly asks this to be checked and documented.
merchant_category_counts = (
    transactions.dropna(subset=["merchant"]).groupby("merchant")["category"].nunique()
)
conflicting_merchants = merchant_category_counts[merchant_category_counts > 1]
print(f"Merchants with conflicting category assignments: {len(conflicting_merchants)}")

occurrences = transactions.dropna(subset=["merchant"])["merchant"].value_counts()
occurrences.to_frame("occurrences")

Merchants with conflicting category assignments: 0


,occurrences
merchant,
Spotify,3
Continente,2
O Pescador,2
Test Merchant,2
Coffee Corner,1
EDP,1
Farmácia Central,1
CP,1
TAP Air,1


**Observed:** 16 unique raw merchant strings, 16 unique casefolded strings —
identical counts, so no casing variant of the same merchant exists. Zero
merchants have conflicting category assignments across their occurrences:
every merchant that repeats (`Spotify` x3, `Continente` x2, `O Pescador` x2,
`Test Merchant` x2) agrees on exactly one category every time.

**Interpretation:** the raw `merchant` and `category` fields are already
reliable for this fixture. PR-011's Critical Rule applies directly —
preserve them, and do not invent a normalisation/aliasing problem that the
evidence does not support. `finance_analytics.enrichment.merchants.MERCHANT_ALIASES`
is therefore empty in production (see its module docstring) rather than
seeded with guessed entries.

## 3. Normalisation Strategy

`enrichment.merchants.normalise_merchant` does exactly two things,
deterministically:

1. **Structural cleanup** — trim surrounding whitespace, collapse internal
   whitespace runs to a single space. Applies to every value; needs no
   evidence, since it can never merge two different merchants.
2. **Curated alias resolution** — a casefolded lookup against an explicit
   table (`MERCHANT_ALIASES`) of raw variant -> canonical name. This is the
   *only* place two differently-formatted strings are allowed to fold into
   one merchant identity, and it is empty here (section 2's finding).

**Explicitly not done:** no fuzzy matching (edit distance, embeddings), no
automatic title-casing (`TAP Air`, `EDP`, `CP`, `MediaMarkt` are correctly
brand-cased in the raw data; title-casing would corrupt them to `Tap Air`,
`Edp`, `Cp`, `Mediamarkt`), no speculative suffix-stripping (e.g.
`".com"`) — nothing in this dataset provides evidence such noise exists,
and PR-011 explicitly warns against assuming similar strings are the same
merchant without evidence.

In [6]:
# The mechanism is still fully implemented and tested with synthetic
# examples (tests/test_enrichment_merchants.py), since real bank CSV
# exports commonly need it even though this fixture does not.
synthetic_aliases = {"netflix": "Netflix", "netflix.com": "Netflix"}
for example in ["NETFLIX", "netflix.com", "  Netflix  ", "Netflix"]:
    print(f"{example!r:20s} -> {normalise_merchant(example, aliases=synthetic_aliases)!r}")

'NETFLIX'            -> 'Netflix'
'netflix.com'        -> 'Netflix'
'  Netflix  '        -> 'Netflix'
'Netflix'            -> 'Netflix'


## 4. Before/After Merchant Statistics

Running the real, production `normalise_merchant` (empty alias table) over
every raw merchant value confirms the section 2/3 conclusion empirically: on
this fixture, normalisation is a no-op.

In [7]:
normalised_merchants = raw_merchants.apply(normalise_merchant)
changed = raw_merchants[raw_merchants != normalised_merchants]

print(f"Unique raw merchants:        {raw_merchants.nunique()}")
print(f"Unique normalised merchants: {normalised_merchants.nunique()}")
print(f"Rows where normalisation changed the value: {len(changed)}")
print(f"Number of merchant aliases applied (production table): {len(MERCHANT_ALIASES)}")

Unique raw merchants:        16
Unique normalised merchants: 16
Rows where normalisation changed the value: 0
Number of merchant aliases applied (production table): 0


**Before/after:** 16 unique raw merchants -> 16 unique normalised merchants,
0 values changed by normalisation, 0 aliases applied. This is expected, not
a bug: it is the direct, quantified consequence of section 2's finding that
the raw data has nothing to normalise. See section 9 (Implications) for
what this means for PR-009/PR-010.

## 5. Category Distribution

`enrichment.categories.KNOWN_CATEGORIES` is not a new taxonomy — it is
exactly the set of category values already used in this project's data
(`docs/project/02_DOMAIN_MODEL.md`: categories "should remain stable to
preserve historical analytics"), plus the explicit fallback
`"Uncategorised"` for a transaction none of the rules can place.

In [8]:
print(f"Known categories ({len(KNOWN_CATEGORIES)}):")
for category in sorted(KNOWN_CATEGORIES):
    print(f"  - {category}")
print(f"Fallback category: {FALLBACK_CATEGORY!r}")

Known categories (10):
  - Cash
  - Food & Dining
  - Groceries
  - Health
  - Income
  - Shopping
  - Subscriptions
  - Transport
  - Travel
  - Utilities
Fallback category: 'Uncategorised'


In [9]:
category_counts = transactions["category"].value_counts()
fig = px.bar(
    category_counts.sort_values(ascending=True),
    orientation="h",
    title="Raw Transaction Count by Category",
    labels={"value": "Transaction count", "category": ""},
)
fig.update_layout(showlegend=False)
fig.show()

**Observed:** 10 distinct category values are used across the 22 raw rows,
none blank, and every one of them already belongs to `KNOWN_CATEGORIES` —
confirmed programmatically in section 7 below, where the categorisation
pipeline is actually run end-to-end.

## 6. Unknown / Uncertain Transactions

PR-011 requires that a transaction is never forced into a category it
isn't supported by evidence for. Two synthetic examples make the fallback
tier concrete (real fixture data has none — see section 7):

In [10]:
unknown_merchant_example = categorise(
    normalise_merchant("Totally New Shop"), raw_category=None, description=None
)
unrecognised_category_example = categorise(
    normalise_merchant("Totally New Shop"), raw_category="Misc", description=None
)

for label, result in [
    ("Unknown merchant, no category, no description", unknown_merchant_example),
    ("Unknown merchant, unrecognised raw category", unrecognised_category_example),
]:
    print(f"{label}:")
    print(f"  category={result.category!r} confidence={result.confidence} method={result.method!r}")

Unknown merchant, no category, no description:
  category='Uncategorised' confidence=0.0 method='fallback'
Unknown merchant, unrecognised raw category:
  category='Uncategorised' confidence=0.0 method='fallback'


Both resolve to `"Uncategorised"` with `category_confidence=0.0` and
`categorisation_method="fallback"` — an explicit, honest outcome rather
than a guessed category. `category_confidence` is a fixed score per method
tier (`CONFIDENCE_BY_METHOD`), not a calibrated probability (PR-011,
Unknown/Uncertain Transactions).

In [11]:
pd.Series(CONFIDENCE_BY_METHOD, name="category_confidence").to_frame()

,category_confidence
known_merchant_rule,1.00
known_description_rule,0.75
existing_category,0.50
fallback,0.00


## 7. Categorisation Strategy

`enrichment.categories.categorise` evaluates four tiers in order, first
match wins:

```
Known merchant rule        (MERCHANT_CATEGORY_RULES)
        |
Known description rule     (DESCRIPTION_KEYWORD_RULES)
        |
Existing trusted category  (raw category, if in KNOWN_CATEGORIES)
        |
Fallback                   ("Uncategorised")
```

`MERCHANT_CATEGORY_RULES` is seeded entirely from this fixture's own
unambiguous merchant -> category history (section 2: zero conflicting
merchants), excluding `"Test Merchant"` (QA fixture, not a real merchant —
same exclusion PR-008's EDA and PR-010 already applied). Its value is not
re-deriving what a row already states, but making that knowledge reusable
for a *different* row from the same merchant with a missing/blank category
— exactly what happens to `transaction_id=21` below.

`DESCRIPTION_KEYWORD_RULES` has exactly one entry, `"subscription" ->
"Subscriptions"`, because it is the only pattern in this dataset with
cross-merchant evidence: both `Spotify` and `Netflix` rows carry the
description "Monthly subscription" and both are `Subscriptions`. Other
single-occurrence phrases (e.g. "Monthly salary") are *not* codified as
rules — one observation is not enough evidence a phrase generalises (see
section 8, Limitations).

In [12]:
print("Known merchant rules:")
for merchant, category in sorted(MERCHANT_CATEGORY_RULES.items()):
    print(f"  {merchant!r:20s} -> {category}")
print()
print("Known description rules:")
for keyword, category in DESCRIPTION_KEYWORD_RULES.items():
    print(f"  {keyword!r:20s} -> {category}")

Known merchant rules:
  'atm'                -> Cash
  'café central'       -> Food & Dining
  'coffee corner'      -> Food & Dining
  'continente'         -> Groceries
  'cp'                 -> Transport
  'edp'                -> Utilities
  'employer payroll'   -> Income
  'farmácia central'   -> Health
  'freelance client'   -> Income
  'galp'               -> Transport
  'mediamarkt'         -> Shopping
  'netflix'            -> Subscriptions
  'o pescador'         -> Food & Dining
  'spotify'            -> Subscriptions
  'tap air'            -> Travel

Known description rules:
  'subscription'       -> Subscriptions


In [13]:
results = enrich_transactions(transactions)
enriched = pd.DataFrame([asdict(r) for r in results])

print(f"Raw rows: {len(transactions)}  |  Enriched (deduplicated by id): {len(enriched)}")
enriched[
    [
        "transaction_id",
        "raw_merchant",
        "normalised_merchant",
        "raw_category",
        "category",
        "category_confidence",
        "categorisation_method",
    ]
]

Raw rows: 22  |  Enriched (deduplicated by id): 21


,transaction_id,raw_merchant,normalised_merchant,raw_category,category,category_confidence,categorisation_method
0,1,Coffee Corner,Coffee Corner,Food & Dining,Food & Dining,1.0,known_merchant_rule
1,2,Continente,Continente,Groceries,Groceries,1.0,known_merchant_rule
2,3,Spotify,Spotify,Subscriptions,Subscriptions,1.0,known_merchant_rule
3,4,O Pescador,O Pescador,Food & Dining,Food & Dining,1.0,known_merchant_rule
4,5,EDP,EDP,Utilities,Utilities,1.0,known_merchant_rule
5,6,Farmácia Central,Farmácia Central,Health,Health,1.0,known_merchant_rule
6,7,CP,CP,Transport,Transport,1.0,known_merchant_rule
7,8,TAP Air,TAP Air,Travel,Travel,1.0,known_merchant_rule
8,9,MediaMarkt,MediaMarkt,Shopping,Shopping,1.0,known_merchant_rule
9,10,Café Central,Café Central,Food & Dining,Food & Dining,1.0,known_merchant_rule


In [14]:
enriched["categorisation_method"].value_counts().to_frame("transaction_count")

,transaction_count
categorisation_method,
known_merchant_rule,18
existing_category,3


**Results on the real (deduplicated) fixture — 21 transactions:**

- **18 via `known_merchant_rule`** (`confidence=1.0`) — every transaction
  whose merchant is one of the 15 known merchants.
- **3 via `existing_category`** (`confidence=0.5`) — the two `Test
  Merchant` QA rows (`id=19,20`) and the missing-merchant row (`id=21`).
  None of these merchants are in `MERCHANT_CATEGORY_RULES` (`Test Merchant`
  deliberately excluded; `id=21` has no merchant at all), and no
  description matched, so each falls through to its own already-trusted
  raw `category`.
- **0 via `known_description_rule`** — every row that could use it (the
  `Spotify`/`Netflix` "Monthly subscription" rows) already resolves at the
  merchant-rule tier first, since the merchant rule is checked before the
  description rule.
- **0 via `fallback`** — nothing in this fixture is truly uncategorisable.

**`id=21` (missing merchant) is the concrete value-add of the priority
chain over "just trust the raw category":** its merchant cannot be
normalised (`normalised_merchant=None`), but it is still safely categorised
as `Shopping` via tier 3, with an explicit `existing_category` method and
`0.5` confidence rather than a silently-dropped or force-guessed row.

In [15]:
enriched.loc[enriched["transaction_id"] == "21"]

,transaction_id,raw_merchant,normalised_merchant,raw_category,raw_description,category,category_confidence,categorisation_method,reason
20,21,NaN,NaN,Shopping,Missing merchant test,Shopping,0.5,existing_category,No merchant or description rule matched; the i...


## 8. Validation Examples

Controlled, deterministic cases — largely synthetic, since the real
fixture (section 7) does not exercise every tier. Same input always
produces the same output.

In [16]:
def show(label: str, transaction_id, raw_merchant, raw_category, raw_description, **kwargs) -> None:
    result = enrich_transaction(
        transaction_id, raw_merchant, raw_category, raw_description, **kwargs
    )
    print(f"{label}")
    print(
        f"  normalised_merchant={result.normalised_merchant!r}  "
        f"category={result.category!r}  confidence={result.category_confidence}  "
        f"method={result.categorisation_method!r}"
    )
    print(f"  reason: {result.reason}")
    print()


synthetic_alias_table = {"netflix": "Netflix", "netflix.com": "Netflix"}

show("Casing difference (production merchant, unusual case)", "v1", "SPOTIFY", None, None)
show("Whitespace difference", "v2", "  Continente   ", None, None)
show(
    "Known alias (synthetic — see section 3; production table has none yet)",
    "v3",
    "netflix.com",
    None,
    None,
    merchant_aliases=synthetic_alias_table,
)
show("Already-normalised merchant", "v4", "Netflix", None, None)
show("Unknown merchant, no other signal", "v5", "Totally New Shop", None, None)
show("Clearly known category (via merchant rule)", "v6", "Spotify", "Something Else", None)
show(
    "Ambiguous / unrecognised category, no merchant match",
    "v7",
    "Totally New Shop",
    "Misc",
    None,
)
show("Missing merchant and description, category present", "v8", None, "Shopping", None)
show("Missing merchant, description and category all missing", "v9", None, None, None)
show(
    "Conflicting signals: known merchant rule vs. contradicting raw category",
    "v10",
    "Netflix",
    "Entertainment",
    None,
)
show(
    "Conflicting signals: description rule vs. contradicting raw category (merchant unknown)",
    "v11",
    None,
    "Shopping",
    "Monthly subscription",
)

Casing difference (production merchant, unusual case)
  normalised_merchant='SPOTIFY'  category='Subscriptions'  confidence=1.0  method='known_merchant_rule'
  reason: 'SPOTIFY' is a known merchant, categorised as Subscriptions.

Whitespace difference
  normalised_merchant='Continente'  category='Groceries'  confidence=1.0  method='known_merchant_rule'
  reason: 'Continente' is a known merchant, categorised as Groceries.

Known alias (synthetic — see section 3; production table has none yet)
  normalised_merchant='Netflix'  category='Subscriptions'  confidence=1.0  method='known_merchant_rule'
  reason: 'Netflix' is a known merchant, categorised as Subscriptions.

Already-normalised merchant
  normalised_merchant='Netflix'  category='Subscriptions'  confidence=1.0  method='known_merchant_rule'
  reason: 'Netflix' is a known merchant, categorised as Subscriptions.

Unknown merchant, no other signal
  normalised_merchant='Totally New Shop'  category='Uncategorised'  confidence=0.0  metho

**Observed:**

- **Casing/whitespace differences** (`v1`, `v2`) do not block the merchant
  rule: rule lookup is casefolded, so `"SPOTIFY"` still matches `"spotify"`
  in `MERCHANT_CATEGORY_RULES`. `normalise_merchant` itself preserves the
  case it was given (`normalised_merchant="SPOTIFY"`, not `"Spotify"`) —
  see section 3 for why blind re-casing is not done.
- **Known alias** (`v3`) shows the mechanism working end-to-end with a
  synthetic alias, distinct from the empty production table.
- **Already-normalised merchant** (`v4`) is unchanged and categorised
  identically to `v3` — same merchant identity, same outcome.
- **Unknown merchant** (`v5`) is safely categorised as `Uncategorised`
  rather than guessed.
- **Clearly known category** (`v6`) shows the merchant rule overriding a
  contradicting raw category, by design (section 7).
- **Ambiguous category** (`v7`) — `"Misc"` is not in `KNOWN_CATEGORIES`, so
  it is not trusted; falls back.
- **Missing merchant + description** (`v8`, `v9`) show the graceful
  degradation chain: `v8` still resolves via the trusted raw category,
  `v9` has genuinely nothing to go on and is an honest `Uncategorised`.
- **Conflicting signals** (`v10`, `v11`) are resolved by priority order,
  not by "most recent" or "most specific" heuristics — whichever tier is
  reached first wins, and `reason` always names which one.

## 9. Limitations

- **No evidence of merchant aliasing, casing or punctuation problems in
  this fixture.** `MERCHANT_ALIASES` and the alias-folding path are
  implemented and unit-tested (`tests/test_enrichment_merchants.py`) but
  unexercised by real data here — validated only synthetically (section 3,
  8). A future, larger or real-world dataset may reveal genuine aliases;
  they should be added to `MERCHANT_ALIASES` only with the same kind of
  evidence this notebook required (a documented, observed variant), not
  speculatively.
- **`MERCHANT_CATEGORY_RULES` is seeded from 15 merchants, each with 1-3
  observed occurrences.** "Unambiguous" here means "never observed with a
  conflicting category in this fixture", not "verified against a large or
  independent sample" — the same small-sample caveat PR-009's and PR-010's
  notebooks already apply to this dataset.
- **`DESCRIPTION_KEYWORD_RULES` has exactly one entry**, deliberately. Other
  candidate phrases ("salary", "withdrawal", "bill") each appear only once
  in this fixture — one observation is evidence, but not enough to
  generalise into a rule with confidence, unlike "subscription" which
  repeats identically across two different merchants (section 7).
- **`category_confidence` is an ordinal ranking, not a calibrated
  probability.** `CONFIDENCE_BY_METHOD`'s four fixed values (1.0 / 0.75 /
  0.5 / 0.0) have not been fit or validated against labelled outcomes —
  see PR-011, Unknown/Uncertain Transactions.
- **The taxonomy (`KNOWN_CATEGORIES`) is exactly the 10 categories already
  present in this fixture.** It has not been validated against a broader,
  real-world category vocabulary; a real user's data will likely need
  categories this fixture does not exercise.
- **Small, synthetic, single-user fixture** (22 raw rows, 21 after
  deduplication) — the same limitation notebooks 02-04 already document.
  Nothing here has been validated at production scale or against a second
  user's data.

## 10. Implications for Future Analytics

**Quantified impact on this fixture: none, by design.** Section 4 showed
normalisation changes zero merchant values here — the raw data was already
reliable, exactly what PR-011's Critical Rule expects this notebook to
check for before building anything more elaborate. Category assignment
(section 7) fully agrees with the raw `category` column everywhere it was
already present, and adds a category to the one row (`id=21`) that would
otherwise have needed its merchant to be present to be trusted.

**Relationship with PR-009 / PR-010 (Recurring/Anomaly detection).** Both
of those modules currently key their historical baselines/candidates by the
*raw* `merchant` column. Since normalisation is a verified no-op on this
fixture, swapping to `enrichment.merchants.normalise_merchant`'s output
would not change a single one of their results here — there is no evidence
of a correctness issue to justify refactoring them (PR-011: "Only refactor
previous detectors if a clear correctness issue is discovered"). Neither
`anomalies/` nor `recurring/` is modified by this PR.

**What a future version *should* do:** once a real or larger dataset
provides evidence of actual merchant-name variants, `anomalies.features`
and `recurring.features` should key their grouping by
`enrichment.merchants.normalise_merchant(merchant)` instead of the raw
`merchant` string — otherwise two spellings of the same merchant would
silently split into two separate (and each thinner) histories/candidates.
That is a follow-up for whichever PR first has that evidence, not this one.

**Reusability.** `enrichment.models.enrich_transactions` takes a raw
transaction `DataFrame` and returns one `EnrichedTransaction` per
transaction — no dependency on `anomalies`/`recurring`, no assumption about
what consumes it next. It is the natural input for a future Insights Engine
(e.g. "your spending in `Uncategorised` transactions this month"), a
category-level dashboard view, or an improved recurring-detection candidate
table keyed by normalised merchant.

## Out of Scope — Confirmation

Not implemented in this notebook or in `finance_analytics.enrichment`: a
production ML categorisation model, LLM categorisation (every `reason`
string above is a fixed template — see `enrichment/explanations.py`),
embedding-based or fuzzy merchant matching, displaying categories in
Android, persisting enriched data in Room, an API, recommendations, or
financial advice. `finance_analytics.anomalies` and
`finance_analytics.recurring` are unmodified by this notebook — see
section 10.